In [1]:
!pip install selenium requests pandas Pillow beautifulsoup4 lxml

In [2]:
import time, re, requests, pandas as pd, os
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import shutil

In [3]:
options = Options()
options.binary_location = "/usr/bin/chromium"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
chromedriver_path = shutil.which("chromedriver") or "/usr/lib/chromium/chromedriver"
driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
print("Driver ready ✅")

Driver ready ✅


In [4]:
driver.get("https://www.houzz.com/photos/home-office-ideas-phbr0-bp~t_732")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
time.sleep(4)
print("Page loaded ✅  Title:", driver.title)

Page loaded ✅  Title: 75 Home Office Ideas You'll Love - April, 2026 | Houzz


In [5]:
def collect_page(driver, room_type):
    for scroll in range(0, 6000, 600):
        driver.execute_script(f"window.scrollTo(0, {scroll});")
        time.sleep(0.3)
    time.sleep(2)

    soup    = BeautifulSoup(driver.page_source, "lxml")
    results = []
    seen    = set()

    SKIP = ("save","view","add to","sign in","join","explore","browse",
            "see more","load more","similar","get","find","discover",
            "shop","click","learn","read")

    ROOM_KEYS = {"floor","wall","ceiling","kitchen","bedroom","bathroom",
                 "living","dining","office","cabinet","countertop","island",
                 "design","photo","example","style","space","room","layout",
                 "storage","renovation","remodel","transitional","modern",
                 "traditional","contemporary","farmhouse","light","window"}

    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if not src.startswith("http"): continue

        m = re.search(r"-w(\d+)-", src)
        if not m or int(m.group(1)) < 300: continue
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue

        caption = None
        node = img

        for _ in range(6):
            node = node.parent
            if node is None: break

            for child in node.find_all(["p","span","div","a"], recursive=False):
                text = child.get_text(separator=" ", strip=True)
                words = text.split()

                if len(words) < 10: continue
                if text.lower().startswith(SKIP): continue

                lower_count = sum(1 for w in words if w.islower())
                if lower_count < 5: continue

                if not set(text.lower().split()) & ROOM_KEYS: continue

                caption = text
                break

            if caption: break

        if not caption: continue

        title = (img.get("alt") or "").strip() or " ".join(caption.split()[:6])
        seen.add(src)
        results.append({"title":title,"description":caption,
                        "image_url":src,"room_type":room_type})
    return results

In [6]:
page_data = collect_page(driver, "home-office")
print(f"Page 1: {len(page_data)} good captions")
for item in page_data[:5]:
    nw = len(item["description"].split())
    print(f"  [{nw} words] {item['description'][:85]}")

Page 1: 40 good captions
  [11 words] Craft room - huge modern craft room idea in New York
  [23 words] This kitchen renovation consisted of adding a built-in desk. Photography (c) Jeffrey 
  [26 words] Photos by Cristopher Nolasco Inspiration for a coastal freestanding desk medium tone 
  [95 words] We created this office in an exclusive home in the Lansdowne community located near L
  [22 words] Study room - large traditional freestanding desk medium tone wood floor and brown flo


In [7]:
URLS  = ['https://www.houzz.com/photos/home-office-ideas-phbr0-bp~t_732']
PAGES = 400
TARGET = 4000
all_data = []
all_seen = set()
print("Room:", "home-office", "| Target:", TARGET)

for ui, base_url in enumerate(URLS):
    style = base_url.split("~s_")[-1] if "~s_" in base_url else "main"
    print(f"\n  URL {ui+1}/{len(URLS)} — {style}")
    empty = 0
    for page in range(1, PAGES+1):
        if len(all_data) >= TARGET:
            print("  ✅ Target reached"); break
        driver.get(f"{base_url}?pg={page}")
        print(f"    Page {page:03d}", end=" → ")
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
        except:
            print("TIMEOUT"); empty += 1
            if empty >= 3: print("  next URL"); break
            continue
        new = [i for i in collect_page(driver, "home-office") if i["image_url"] not in all_seen]
        for i in new: all_seen.add(i["image_url"])
        all_data.extend(new)
        print(f"{len(new):3d} new | total: {len(all_data)}")
        empty = 0 if new else empty + 1
        if empty >= 5: print("  No new — next URL"); break
        time.sleep(1.5)
    if len(all_data) >= TARGET: break

print(f"\n✅ Done. Total: {len(all_data)} images")

Room: home-office | Target: 4000

  URL 1/1 — main
    Page 001 →  40 new | total: 40
    Page 002 →   0 new | total: 40
    Page 003 →  20 new | total: 60
    Page 004 →  20 new | total: 80
    Page 005 →  20 new | total: 100
    Page 006 →  20 new | total: 120
    Page 007 →  20 new | total: 140
    Page 008 →  20 new | total: 160
    Page 009 →  20 new | total: 180
    Page 010 →  20 new | total: 200
    Page 011 →  20 new | total: 220
    Page 012 →  20 new | total: 240
    Page 013 →  20 new | total: 260
    Page 014 →  20 new | total: 280
    Page 015 →  20 new | total: 300
    Page 016 →  20 new | total: 320
    Page 017 →  20 new | total: 340
    Page 018 →  20 new | total: 360
    Page 019 →  20 new | total: 380
    Page 020 →  20 new | total: 400
    Page 021 →  20 new | total: 420
    Page 022 →  20 new | total: 440
    Page 023 →  20 new | total: 460
    Page 024 →  20 new | total: 480
    Page 025 →  20 new | total: 500
    Page 026 →  20 new | total: 520
    Page 027 →  2

In [8]:
driver.quit()
print('Driver closed ✅')

Driver closed ✅


In [9]:
df = pd.DataFrame(all_data).drop_duplicates(subset=["image_url"]).reset_index(drop=True)
df["wc"] = df["description"].str.split().str.len()
print(f"Total: {len(df)} | Min words: {df['wc'].min()} | Avg: {df['wc'].mean():.1f} | All≥10: {(df['wc']>=10).all()} ✅")
df.head()

Total: 4005 | Min words: 10 | Avg: 105.5 | All≥10: True ✅


,title,description,image_url,room_type,wc
0,Office & Craft Rooms,Craft room - huge modern craft room idea in Ne...,https://st.hzcdn.com/fimgs/735123760cd47b81_26...,home-office,11
1,Built-in Desk,This kitchen renovation consisted of adding a ...,https://st.hzcdn.com/fimgs/d2e148000480d5f7_60...,home-office,23
2,Beach Chic Farmhouse,Photos by Cristopher Nolasco Inspiration for a...,https://st.hzcdn.com/fimgs/5981c55e0d4204aa_41...,home-office,26
3,Custom home office,We created this office in an exclusive home in...,https://st.hzcdn.com/fimgs/8a51dfd704cd1a2e_64...,home-office,95
4,Painted Brick with Southern Charm,Study room - large traditional freestanding de...,https://st.hzcdn.com/fimgs/192130620b880fe9_91...,home-office,22


In [10]:
os.makedirs("images/home-office", exist_ok=True)
H = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}
lp, fail = [], 0
for i, row in df.iterrows():
    fname = "images/home-office/home-office_" + str(i).zfill(5) + ".jpg"
    try:
        r = requests.get(row["image_url"], headers=H, timeout=15)
        if r.status_code == 200:
            Image.open(BytesIO(r.content)).convert("RGB").save(fname)
            lp.append(fname)
        else: lp.append(None); fail+=1
    except: lp.append(None); fail+=1
    time.sleep(0.4)
df["local_path"] = lp
print(f"Downloaded: {len(df)-fail} | Failed: {fail}")

Downloaded: 4005 | Failed: 0


In [11]:
df = df.dropna(subset=["local_path"]).drop(columns=["wc"],errors="ignore").reset_index(drop=True)
print(f"Final: {len(df)} images")
df.head()

Final: 4005 images


,title,description,image_url,room_type,local_path
0,Office & Craft Rooms,Craft room - huge modern craft room idea in Ne...,https://st.hzcdn.com/fimgs/735123760cd47b81_26...,home-office,images/home-office/home-office_00000.jpg
1,Built-in Desk,This kitchen renovation consisted of adding a ...,https://st.hzcdn.com/fimgs/d2e148000480d5f7_60...,home-office,images/home-office/home-office_00001.jpg
2,Beach Chic Farmhouse,Photos by Cristopher Nolasco Inspiration for a...,https://st.hzcdn.com/fimgs/5981c55e0d4204aa_41...,home-office,images/home-office/home-office_00002.jpg
3,Custom home office,We created this office in an exclusive home in...,https://st.hzcdn.com/fimgs/8a51dfd704cd1a2e_64...,home-office,images/home-office/home-office_00003.jpg
4,Painted Brick with Southern Charm,Study room - large traditional freestanding de...,https://st.hzcdn.com/fimgs/192130620b880fe9_91...,home-office,images/home-office/home-office_00004.jpg


In [12]:
df.to_csv("home_office_dataset.csv", index=False)
print("Saved → home_office_dataset.csv ✅")

Saved → home_office_dataset.csv ✅


In [13]:
pd.read_csv("home_office_dataset.csv")

,title,description,image_url,room_type,local_path
0,Office & Craft Rooms,Craft room - huge modern craft room idea in Ne...,https://st.hzcdn.com/fimgs/735123760cd47b81_26...,home-office,images/home-office/home-office_00000.jpg
1,Built-in Desk,This kitchen renovation consisted of adding a ...,https://st.hzcdn.com/fimgs/d2e148000480d5f7_60...,home-office,images/home-office/home-office_00001.jpg
2,Beach Chic Farmhouse,Photos by Cristopher Nolasco Inspiration for a...,https://st.hzcdn.com/fimgs/5981c55e0d4204aa_41...,home-office,images/home-office/home-office_00002.jpg
3,Custom home office,We created this office in an exclusive home in...,https://st.hzcdn.com/fimgs/8a51dfd704cd1a2e_64...,home-office,images/home-office/home-office_00003.jpg
4,Painted Brick with Southern Charm,Study room - large traditional freestanding de...,https://st.hzcdn.com/fimgs/192130620b880fe9_91...,home-office,images/home-office/home-office_00004.jpg
...,...,...,...,...,...
4000,Currant Office Collection,Greenington’s Folio file cabinet features clea...,https://st.hzcdn.com/fimgs/acb172f8020451bb_39...,home-office,images/home-office/home-office_04000.jpg
4001,Office Barn Doors,Home office - mid-sized country home office id...,https://st.hzcdn.com/fimgs/13716bb2090b1fdc_05...,home-office,images/home-office/home-office_04001.jpg
4002,Modern Las Vegas Home,Lucky Wenzel Inspiration for a large modern fr...,https://st.hzcdn.com/fimgs/21b1e7770882a360_55...,home-office,images/home-office/home-office_04002.jpg
4003,Top Down Bottom Up (TDBU) Honeycomb shades,All Filters Style Contemporary Modern Traditio...,https://st.hzcdn.com/fimgs/32e19d860c99868d_92...,home-office,images/home-office/home-office_04003.jpg
